# Redraw route figures from saved `*_routes.pt`

Pure-visualization notebook. **Reads only**: every `artifacts/results/*_routes.pt`
produced by the experiment notebooks (`evaluation_seeded_lc_bco_mumford0.ipynb`,
`lc_improvement_training.ipynb`, the benchmark / NSGA-II / MACSA sections) is
reloaded and re-rendered here — no experiment is re-run, no `.pt` / `.csv` file
is overwritten.

For each payload two figures are drawn:

1. **Diff vs reference** — same layout as the experiment notebooks: cell 0 is the
   reference (first run in the file, typically the seed / initial route set),
   cells 1.. are drawn as a route-diff vs the reference.
2. **Plain routes** — companion figure where every run is drawn as a plain
   route set (no diff, no diff legend). The per-panel subtitle leads with the
   weighted `cost=...` so the cost is visible alongside each map.

Both figures use the same `:.2f` fixed-point formatting for all metric numbers
(no `e` notation) and skip the per-route node-sequence tables (controlled by
`SHOW_ROUTE_SEQUENCE_TABLES` in `eval_lib/params.py`, default `False`).

In [ ]:
import matplotlib.pyplot as plt

from eval_lib import *  # noqa: F401,F403
from eval_lib import (load_route_results, render_route_comparison_figure,
                      render_route_set_figure, RESULTS_DIR)

# Discover every saved route payload. Filenames end with `_routes.pt` -- strip
# that suffix to recover the experiment name accepted by `load_route_results`.
payload_paths = sorted(RESULTS_DIR.glob('*_routes.pt'))
experiment_names = [p.name[:-len('_routes.pt')] for p in payload_paths]
print(f'Found {len(experiment_names)} saved route payloads:')
for name in experiment_names:
    print(f'  {name}')

In [ ]:
# Render two figures per payload: diff-vs-reference and plain-routes.
# Re-run this cell after editing eval_lib/figures.py to refresh every figure.
for name in experiment_names:
    results, coords, street_adj = load_route_results(name)
    if not results:
        print(f'[skip] {name}: empty payload')
        continue

    # 1) Diff vs the first run (reference). Only meaningful when len(results) >= 2.
    if len(results) >= 2:
        fig_diff = render_route_comparison_figure(
            results[0], results[1:], coords, street_adj,
            title=f'{name} (diff vs {results[0].label or "reference"})',
            ncols=3)
        plt.show()
        plt.close(fig_diff)
    else:
        print(f'[note] {name}: only one run stored, skipping diff figure')

    # 2) Plain routes -- every run drawn as a plain route set, cost in subtitle.
    fig_plain = render_route_set_figure(
        results, coords, street_adj,
        title=f'{name} (plain routes)',
        ncols=3)
    plt.show()
    plt.close(fig_plain)